# Phase B: Dynamic-Sampling RL Training (Google Colab)

Trains the RL-boosted Frenet planner where the agent's action **also shapes the sampling grid** (longitudinal horizon + lateral span), per the project proposal.

**Read this first:**
- This stack is pinned to **Python 3.10** (the `frenetix` wheel). Colab ships 3.11+, so Cell 2 installs Python 3.10 via `condacolab` and **restarts the runtime** — that is expected; just run the next cell after it reconnects.
- The bottleneck is the **CPU** Frenet simulation, not the GPU. Colab GPU helps only the small LSTM policy. Full 7M-step training (paper: ~24h on a workstation) is **not feasible** here. Use a reduced budget (`TOTAL_TIMESTEPS` below) — aim for a proof-of-concept agent, not paper-matching numbers.
- Checkpoints go to **Google Drive** so a disconnect doesn't lose progress; re-running resumes-by-restart from the last saved model if you wire it up (see final note).

**Before running:** zip your `Frenetix-RL` folder and upload it to Google Drive at `MyDrive/Frenetix-RL.zip` (or adjust `DRIVE_REPO_ZIP`).

In [1]:
# Cell 1 — confirm a GPU is attached (Runtime > Change runtime type > GPU)
!nvidia-smi

Wed Jun  3 08:37:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — install Python 3.10 via condacolab. THIS RESTARTS THE RUNTIME.
# After it reconnects, skip this cell and continue from Cell 3.
!pip install -q condacolab
import condacolab
condacolab.install()  # runtime restarts here

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


In [ ]:
# Cell 3 — verify Python 3.10 and create a 3.10 env
import sys; print('python:', sys.version)
!conda create -y -n frl python=3.10 >/dev/null 2>&1 && echo 'env frl created'

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
env frl created


: 

In [ ]:
# Cell 4 — system build deps for frenetix C++ core
!apt-get -qq update && apt-get -qq install -y cmake build-essential libeigen3-dev libboost-all-dev libomp-dev libgl1 libglib2.0-0 >/dev/null && echo 'apt deps ok'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
apt deps ok


: 

: 

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: apt-extracttemplates failed: No such file or directory
Extracting templates from packages: 28%

In [1]:
# Cell 5 — mount Drive and unpack the repo (idempotent: nukes /content/work first
# so the logs symlink from a previous run doesn't collide with the zip's logs/ entry)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_REPO_ZIP = '/content/drive/MyDrive/Frenetix-RL.zip'
import os, shutil, zipfile
shutil.rmtree('/content/work', ignore_errors=True)
os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile(DRIVE_REPO_ZIP) as z:
    z.extractall('/content/work')
# adjust if your zip nests the folder differently
REPO = '/content/work/Frenetix-RL' if os.path.isdir('/content/work/Frenetix-RL') else '/content/work'
print('REPO =', REPO); print(sorted(os.listdir(REPO))[:20])

Mounted at /content/drive
REPO = /content/work/Frenetix-RL
['.dockerignore', '.git', '.gitignore', 'Dockerfile', 'LICENSE', 'README.md', 'TRAINING_LINUX.md', '__pycache__', 'analysis', 'best_model-700k', 'best_model_b', 'best_model_hp', 'colab', 'configurations', 'execute.py', 'frenetix_rl', 'frl_hp_100k_logs_test', 'frl_phaseB_logs_test', 'logs', 'logs_dp']


In [2]:
# Cell 6 — install the project into the 3.10 env (GPU torch + pinned deps)
%cd {REPO}
!conda run -n frl pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!conda run -n frl pip install -q . && conda run -n frl pip install -q 'matplotlib<3.8' 'scipy<1.14'
!conda run -n frl python -c "import frenetix, cr_scenario_handler, frenetix_rl, torch; print('imports OK | cuda:', torch.cuda.is_available())"

/content/work/Frenetix-RL
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
frenetix-motion-planner 2024.1 requires matplotlib<4.0.0,>=3.8.1, but you have matplotlib 3.7.5 which is incompatible.

imports OK | cuda: True



In [3]:
# Cell 7 — wire logs/best_model/intermediate_model into Drive so disconnects don't lose progress.
# (Phase B config — action_type, reward_sampling_efficiency, total_timesteps=100000, num_envs=2,
#  and the plotting/GIF flags — is now committed in configs.yaml + visualization.yaml,
#  so this cell no longer patches anything.)
import os, shutil, yaml
DRIVE_LOGS = '/content/drive/MyDrive/frl_phaseB_logs'
os.makedirs(DRIVE_LOGS, exist_ok=True)
logs_link = os.path.join(REPO, 'logs')
if not os.path.islink(logs_link):
    shutil.rmtree(logs_link, ignore_errors=True)
    os.symlink(DRIVE_LOGS, logs_link)

# Confirm Phase B settings are present in the YAML.
cfg = yaml.safe_load(open(os.path.join(REPO, 'frenetix_rl/gym_environment/configs.yaml')))['env_configs']
print('action_type            =', cfg['action_configs']['action_type'])
print('reward_sampling_eff    =', cfg['reward_configs']['dense_reward']['reward_sampling_efficiency'])
print('total_timesteps        =', cfg['training_configs']['total_timesteps'])
print('num_envs               =', cfg['training_configs']['num_envs'])
print('logs ->', os.readlink(logs_link))

action_type            = weights_and_sampling
reward_sampling_eff    = -0.0002
total_timesteps        = 100000
num_envs               = 2
logs -> /content/drive/MyDrive/frl_phaseB_logs


In [8]:
# Cell 8 — train.
# --no-capture-output: stream stdout/stderr live (otherwise conda buffers them
#                      until the subprocess exits, hiding the tqdm bar).
# python -u           : unbuffered, so each tqdm refresh flushes immediately.
%cd {REPO}
!conda run --no-capture-output -n frl python -u train.py

/content/work/Frenetix-RL

CondaError: KeyboardInterrupt

Traceback (most recent call last):
  File "/content/work/Frenetix-RL/train.py", line 11, in <module>
    import frenetix_rl.utils.helper_functions as hf
  File "/content/work/Frenetix-RL/frenetix_rl/utils/helper_functions.py", line 7, in <module>
    from frenetix_rl.gym_environment.environment.agent_env import AgentEnv
  File "/content/work/Frenetix-RL/frenetix_rl/gym_environment/environment/agent_env.py", line 24, in <module>
    from commonroad.common.file_reader import CommonRoadFileReader
  File "/usr/local/envs/frl/lib/python3.10/site-packages/commonroad/common/file_reader.py", line 5, in <module>
    from commonroad.common.reader.file_reader_protobuf import ProtobufFileReader
  File "/usr/local/envs/frl/lib/python3.10/site-packages/commonroad/common/reader/file_reader_protobuf.py", line 30, in <module>
    from commonroad.scenario_definition.protobuf_format.generated_scripts import commonroad_pb2, util_pb2, \
  File "/usr

In [10]:
# Cell 9 — evaluate the trained dynamic-sampling agent on the held-out
# 55-scenario test set (scenarios_test/, per split_manifest.json).
#
# 1) Symlink logs_phase_b/ -> Drive so eval output survives session restarts
#    (otherwise it lives in /content/work which is wiped on reconnect).
# 2) Run run_phase_b_eval.py (now writes eval_summary.csv incrementally —
#    authoritative per-scenario outcomes, robust to disconnects).
# 3) Extract metrics with success-rate detection that does NOT depend on
#    score_overview.csv: it uses eval_summary.csv if present, else falls back
#    to a logs-length heuristic (>=140 steps = success). Reports the Phase B
#    fields: mean horizon, lateral half-span, and back-computed candidate
#    count (your efficiency-claim number).
#
# Requires: re-upload the repo zip after these fixes were applied locally —
# specifically updated run_phase_b_eval.py and analysis/extract_metrics.py.
import os, shutil
DRIVE_PB = '/content/drive/MyDrive/frl_phaseB_logs_test'
os.makedirs(DRIVE_PB, exist_ok=True)
lp = os.path.join(REPO, 'logs_phase_b')
if not os.path.islink(lp):
    shutil.rmtree(lp, ignore_errors=True)
    os.symlink(DRIVE_PB, lp)
print('logs_phase_b ->', os.readlink(lp))

%cd {REPO}
!conda run --no-capture-output -n frl python -u run_phase_b_eval.py
!conda run -n frl python analysis/extract_metrics.py --logs-dir logs_phase_b --label 'Phase B (test set, dynamic sampling)'

logs_phase_b -> /content/drive/MyDrive/frl_phaseB_logs_test
/content/work/Frenetix-RL
Loaded model: /content/work/Frenetix-RL/logs/best_model/best_model.zip action_space=(7,)
Test scenarios: 55

[1/55] ZAM_Tjunction-1_106_T-1
Scenario Name: ZAM_Tjunction-1_106_T-1
CRITICAL [simulation.py]: Simulating Scenario: ZAM_Tjunction-1_106_T-1
--- Initializing prediction network ---
/usr/local/envs/frl/lib/python3.10/site-packages/prediction/main.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be lo

In [11]:
# Cell 9b — fair-budget HP comparison (same 100k steps, fixed grid)
import os, shutil
DRIVE_HP = '/content/drive/MyDrive/frl_hp_100k_logs_test'
os.makedirs(DRIVE_HP, exist_ok=True)
lp = os.path.join(REPO, 'logs_hp_100k')
if not os.path.islink(lp):
    shutil.rmtree(lp, ignore_errors=True)
    os.symlink(DRIVE_HP, lp)

%cd {REPO}
!conda run --no-capture-output -n frl python -u run_hp_100k_eval.py
!conda run -n frl python analysis/extract_metrics.py --logs-dir logs_hp_100k --label 'HP-100k (fixed grid, fair-budget baseline)'


/content/work/Frenetix-RL
Packed HP model from /content/work/Frenetix-RL/best_model_hp -> /content/work/Frenetix-RL/logs/best_model/best_model_hp_100k.zip
Loaded model: /content/work/Frenetix-RL/logs/best_model/best_model_hp_100k.zip action_space=(5,)
Test scenarios: 55

[1/55] ZAM_Tjunction-1_106_T-1
Scenario Name: ZAM_Tjunction-1_106_T-1
CRITICAL [simulation.py]: Simulating Scenario: ZAM_Tjunction-1_106_T-1
--- Initializing prediction network ---
/usr/local/envs/frl/lib/python3.10/site-packages/prediction/main.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be exe

In [4]:
# Cell 9d — extended-budget weights-only comparison (700k steps, fixed grid).
# best_model-700k/ is a 5-dim weights-only policy (same action type as the HP
# baseline, just trained ~7x longer). run_700k_eval.py overrides
# action_type='weights' and re-packs best_model-700k/ -> logs/best_model/
# best_model_700k.zip on the fly if the zip is missing.
import os, shutil
DRIVE_700K = '/content/drive/MyDrive/frl_700k_logs_test'
os.makedirs(DRIVE_700K, exist_ok=True)
lp = os.path.join(REPO, 'logs_700k')
if not os.path.islink(lp):
    shutil.rmtree(lp, ignore_errors=True)
    os.symlink(DRIVE_700K, lp)

%cd {REPO}
!conda run --no-capture-output -n frl python -u run_700k_eval.py
!conda run -n frl python analysis/extract_metrics.py --logs-dir logs_700k --label 'HP-700k (fixed grid, extended-budget baseline)'


/content/work/Frenetix-RL
Packed 700k model from /content/work/Frenetix-RL/best_model-700k -> /content/work/Frenetix-RL/logs/best_model/best_model_700k.zip
Loaded model: /content/work/Frenetix-RL/logs/best_model/best_model_700k.zip action_space=(5,)
Test scenarios: 55

[1/55] ZAM_Tjunction-1_106_T-1
Scenario Name: ZAM_Tjunction-1_106_T-1
CRITICAL [simulation.py]: Simulating Scenario: ZAM_Tjunction-1_106_T-1
--- Initializing prediction network ---
/usr/local/envs/frl/lib/python3.10/site-packages/prediction/main.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be execu

In [12]:
# Cell 9c — full metric comparison incl. paper Fig.10 timing decomposition.
# Needs the instrumented eval scripts (Cell 9 / 9b / 9d) to have written
# timing_steps.csv into logs_phase_b/, logs_hp_100k/ and logs_700k/.
# Pure post-hoc on logs -- no retraining.
%cd {REPO}
!conda run --no-capture-output -n frl python -u analysis/compare_efficiency.py \
    --run "HP-100k:logs_hp_100k" \
    --run "HP-700k:logs_700k" \
    --run "PhaseB:logs_phase_b"


/content/work/Frenetix-RL

=== HP-100k  (55 scenarios) ===
  success rate           : 41/55 = 74.5%
  calc_time / iter  micro : 503.6 ms   macro: 518.5 ms
  n_traj sampled    micro : 1148        macro: 1144
  ego_risk   micro : 2.407e-04   macro(mean): 3.581e-04   macro(max): 3.080e-03
  obst_risk  micro : 6.430e-05   macro(mean): 8.908e-05   macro(max): 8.325e-04
  -- Fig.10 decomposition (ms/iter, micro) --
     RL prediction       : 2.533
     trajectory bundle   : 503.6   (= calculation_time_s)
     env step (w/ bundle): 600.0
     overall model step  : 602.5   (= predict + step)

=== PhaseB  (55 scenarios) ===
  success rate           : 43/55 = 78.2%
  calc_time / iter  micro : 351.2 ms   macro: 358.2 ms
  n_traj sampled    micro : 851        macro: 849   (horizon-est: 560)
  ego_risk   micro : 2.054e-04   macro(mean): 3.072e-04   macro(max): 2.785e-03
  obst_risk  micro : 5.660e-05   macro(mean): 7.996e-05   macro(max): 8.371e-04
  -- Fig.10 decomposition (ms/iter, micro) --
    

## Notes
- **Resuming after a disconnect:** `train.py` builds a fresh model each run, so a disconnect loses in-RAM progress; the periodic checkpoints in `logs/intermediate_model` and `logs/best_model` are on Drive. To truly resume, modify `train.py` to `RecurrentPPO.load(<checkpoint>, env=...)` instead of constructing a new model when a checkpoint exists. Ask Claude to add this if you need long runs.
- **If `pip install .` fails building `frenetix`:** there is usually a manylinux cp310 wheel on PyPI; the apt deps in Cell 4 cover a source build if not.
- **Sanity smoke test** (optional): `!conda run -n frl env PYTHONPATH={REPO} python analysis/smoke_test_sampling.py` should print `RESULT: PASS`.
- **Comparison for the report:** evaluate the trained dynamic-sampling agent vs the fixed-grid baseline on the same scenarios; key metrics are success/collision, ego & 3rd-party risk, and **mean candidate-trajectory count / calc-time per step** (the efficiency claim).